# 02 — CRS e georreferenciamento da prancha
> **Estudo educacional independente; não substitui o PAE oficial, a Defesa Civil ou laudo de engenharia.**

A prancha informa **SIRGAS 2000 / UTM 22S (EPSG:31982)**. A grade UTM do mapa é **vetorial** no PDF (12 linhas); os rótulos (667000…682000 E; 7431000…7446000 N) identificam cada linha → 36 pontos de controle exatos.

In [1]:
import sys, warnings; warnings.filterwarnings('ignore')
sys.path.insert(0, '../src')
import pandas as pd, numpy as np, matplotlib.pyplot as plt
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 30)
from jurumirim import DISCLAIMER
print(DISCLAIMER)

Estudo educacional independente; não substitui o PAE oficial, a Defesa Civil ou laudo de engenharia.


In [2]:
from jurumirim.georef import fit_transform, section_lines, independent_check
from jurumirim.pae import extract_sections
A, info = fit_transform()
print(A)
{k: v for k, v in info.items() if k != 'control_points'}

[[ 1.05834834e+01 -1.08616171e-10  6.63961223e+05]
 [ 1.06540519e-09 -1.05822116e+01  7.44774897e+06]]


{'n_control_points': 36,
 'rms_m': 0.4746323257524445,
 'max_residual_m': 0.5683261185185984,
 'scale_x_m_per_pt': 10.583483381496134,
 'scale_y_m_per_pt': 10.58221161058286,
 'crs': 'EPSG:31982'}

## Erro de registro (RMS)
O RMS dos resíduos é submétrico — esperado, pois a grade é desenhada em posições exatas. Isso mede **só o registro da grade**, não a precisão do contorno do PAE.

In [3]:
chk = independent_check(A, extract_sections(), section_lines())
s = pd.Series(chk).round(1); print(s.to_string())
print('\nST-001 (brecha) não tem traço próprio: fora da checagem; demais seções: máx =', s.drop('ST-001').max(), 'm')

ST-001    112.6
ST-002      0.1
ST-003      0.9
ST-004      0.0
ST-005      0.2
ST-006      0.4
ST-007      0.6
ST-008      0.1
ST-009      0.6
ST-010      0.6
ST-011      0.3
ST-012      0.5
ST-013      0.5
ST-014      0.6
ST-015      0.2
ST-016      0.4
ST-017      0.1

ST-001 (brecha) não tem traço próprio: fora da checagem; demais seções: máx = 0.9 m


## Por que isso importa
Com a checagem independente (ponto central tabelado × traço de seção desenhado), 16/17 seções coincidem em <1 m — confirma que a transformação e a leitura das coordenadas estão consistentes.

In [4]:
from pyproj import CRS, Transformer
print(CRS.from_epsg(31982).name)
t = Transformer.from_crs(31982, 4326, always_xy=True)
print('Portaria UHE Jurumirim (X 680.478; Y 7.432.283) ->', t.transform(680478, 7432283))

SIRGAS 2000 / UTM zone 22S
Portaria UHE Jurumirim (X 680.478; Y 7.432.283) -> (-49.23639180840052, -23.20869147103888)
